In [13]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import torchvision.utils as vutils
torch.manual_seed(42)

In [14]:
# 数据预处理
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [15]:
# 加载完整数据集并创建 DataLoader
full_dataset = datasets.ImageFolder(root='Data', transform=transform)
dataloader = DataLoader(full_dataset, batch_size=64, shuffle=True)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

当前使用的设备: cuda


In [17]:
# 权重初始化函数
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0)

In [18]:
# 生成器
class Generator(torch.nn.Module):
    def __init__(self, z_dim):
        super(Generator, self).__init__()

        # 使用 Sequential 堆叠反卷积层 (ConvTranspose2d)
        # 输入维度: z_dim (默认 100)
        # 尺寸变化过程: 1x1 -> 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
        self.main = torch.nn.Sequential(
            # Layer 1: 输入噪声 z，输出尺寸 (512, 4, 4)
            torch.nn.ConvTranspose2d(z_dim, 512, kernel_size=4, stride=1, padding=0, bias=False),
            torch.nn.BatchNorm2d(512),
            torch.nn.ReLU(True),

            # Layer 2: 输出尺寸 (256, 8, 8)
            torch.nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(True),

            # Layer 3: 输出尺寸 (128, 16, 16)
            torch.nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(True),

            # Layer 4: 输出尺寸 (64, 32, 32)
            torch.nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(True),

            # Layer 5: 输出尺寸 (3, 64, 64)
            torch.nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1, bias=False),
            # 生成器的最后一层必须是 Tanh，把输出值压扁到 [-1, 1] 以匹配数据预处理中的 Normalize
            torch.nn.Tanh()
        )

    def forward(self, x):
        # 将随机噪声 reshape 成特征图的形式 (batch_size, z_dim, 1, 1)
        x = x.view(x.size(0), -1, 1, 1)
        return self.main(x)

In [19]:
class Discriminator(torch.nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        # 使用 Sequential 堆叠卷积层 (Conv2d)
        # 输入维度: 3通道图片 (3, 64, 64)
        # 尺寸变化过程: 64x64 -> 32x32 -> 16x16 -> 8x8 -> 4x4 -> 1x1
        self.main = torch.nn.Sequential(
            # Layer 1: 输出尺寸 (64, 32, 32)
            # 判别器的第一层通常不加 BatchNorm
            torch.nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.LeakyReLU(0.2, inplace=True),

            # Layer 2: 输出尺寸 (128, 16, 16)
            torch.nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(128),
            torch.nn.LeakyReLU(0.2, inplace=True),

            # Layer 3: 输出尺寸 (256, 8, 8)
            torch.nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(256),
            torch.nn.LeakyReLU(0.2, inplace=True),

            # Layer 4: 输出尺寸 (512, 4, 4)
            torch.nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(512),
            torch.nn.LeakyReLU(0.2, inplace=True),

            # Layer 5: 降维到 (1, 1, 1)
            torch.nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
            # 判别器的最后一层使用 Sigmoid，输出一个 0 到 1 之间的概率值 (真假概率)
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        # 输出的 shape 默认是 (batch_size, 1, 1, 1)，通过 view 展平成 (batch_size, 1)
        output = self.main(x)
        return output.view(-1, 1)

In [20]:
# 实例化模型并推入 GPU
z_dim = 100
G = Generator(z_dim).to(device)
D = Discriminator().to(device)

# 应用权重初始化函数
G.apply(weights_init)
D.apply(weights_init)

Discriminator(
  (main): Sequential(
    (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (12): Sigmoid()
  )
)

In [21]:
# 二元交叉熵损失 (BCE Loss)
criterion = torch.nn.BCELoss()
optimizerD = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [22]:
# 训练循环
# 创建一个文件夹用来保存生成的图片
os.makedirs("GAN_Results", exist_ok=True)
num_epochs = 50
fixed_noise = torch.randn(64, z_dim, 1, 1).to(device)

print("开始训练对抗网络...")
for epoch in range(num_epochs):
    for i, (imgs, _) in enumerate(dataloader):

        # 获取当前 batch 的大小 (最后一个 batch 可能不满 64)
        batch_size = imgs.size(0)
        # 将真实图片推入显卡
        real_imgs = imgs.to(device)

        # 准备真假标签
        # 真实图片的标签全为 1，假图片的标签全为 0
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # ------------------------------------------
        # 训练判别器 D: 目标是 最大化 log(D(x)) + log(1 - D(G(z)))
        # 让 D 努力认出真图(1)，并且认出假图(0)
        # ------------------------------------------
        D.zero_grad()

        # 1. 喂给 D 真实图片，计算 D 认出真图的 Loss
        output_real = D(real_imgs)
        errD_real = criterion(output_real, real_labels)
        errD_real.backward() # 反向传播

        # 2. 生成随机噪声，并用 G 生成假图片
        noise = torch.randn(batch_size, z_dim, 1, 1).to(device)
        fake_imgs = G(noise)

        # 3. 喂给 D 假图片，计算 D 认出假图的 Loss
        # 注意：这里必须使用 fake_imgs.detach()，切断梯度传导到 G，否则会报错
        output_fake = D(fake_imgs.detach())
        errD_fake = criterion(output_fake, fake_labels)
        errD_fake.backward() # 反向传播

        # 更新 D 的参数
        errD = errD_real + errD_fake
        optimizerD.step()

        # ------------------------------------------
        # (B) 训练生成器 G: 目标是 最大化 log(D(G(z)))
        # 让 G 生成的假图尽可能骗过 D，让 D 给出 1 (真) 的评分
        # ------------------------------------------
        G.zero_grad()

        # 喂给 D 刚才生成的假图 (这次不 detach，因为要计算 G 的梯度)
        output_G = D(fake_imgs)#当前的 G 生成的图片在新的 D 下的评分

        # G 的目标是让 D 判定假图为真，所以拿 real_labels (全是1) 作为目标去计算 Loss
        errG = criterion(output_G, real_labels)
        errG.backward() # 反向传播

        # 更新 G 的参数
        optimizerG.step()

        # ------------------------------------------
        # 打印训练进度
        # ------------------------------------------
        if i % 100 == 0:
            print(f'Epoch [{epoch}/{num_epochs}] Batch [{i}/{len(dataloader)}] '
                  f'Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f}')

    # 每跑完一个 Epoch，用固定的噪声生成 64 张图片并保存，观察训练效果
    with torch.no_grad():
        test_images = G(fixed_noise).detach().cpu()
    # Normalize=True 会自动把模型输出的 [-1, 1] 转换回 [0, 1] 并保存为图片
    vutils.save_image(test_images, f"gan_results/epoch_{epoch:03d}.png", normalize=True)

print("训练完成！请前往 gan_results 文件夹查看成果。")

开始训练对抗网络...
Epoch [0/50] Batch [0/994] Loss_D: 1.6746 Loss_G: 4.1761
Epoch [0/50] Batch [100/994] Loss_D: 1.4752 Loss_G: 11.4269
Epoch [0/50] Batch [200/994] Loss_D: 0.8827 Loss_G: 10.9881
Epoch [0/50] Batch [300/994] Loss_D: 1.3286 Loss_G: 4.1025
Epoch [0/50] Batch [400/994] Loss_D: 0.8462 Loss_G: 4.8084
Epoch [0/50] Batch [500/994] Loss_D: 2.0987 Loss_G: 13.8893
Epoch [0/50] Batch [600/994] Loss_D: 0.1941 Loss_G: 6.5638
Epoch [0/50] Batch [700/994] Loss_D: 0.5060 Loss_G: 6.4637
Epoch [0/50] Batch [800/994] Loss_D: 0.7703 Loss_G: 2.8919
Epoch [0/50] Batch [900/994] Loss_D: 0.3917 Loss_G: 5.9411
Epoch [1/50] Batch [0/994] Loss_D: 0.6861 Loss_G: 9.3636
Epoch [1/50] Batch [100/994] Loss_D: 0.5103 Loss_G: 4.7678
Epoch [1/50] Batch [200/994] Loss_D: 0.4321 Loss_G: 4.9597
Epoch [1/50] Batch [300/994] Loss_D: 1.2819 Loss_G: 11.4817
Epoch [1/50] Batch [400/994] Loss_D: 0.3594 Loss_G: 5.0104
Epoch [1/50] Batch [500/994] Loss_D: 0.9023 Loss_G: 6.5022
Epoch [1/50] Batch [600/994] Loss_D: 0.2010 